In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]  # adjust if notebook is nested deeper
sys.path.append(str(PROJECT_ROOT))

In [5]:
from shared.database.client import SQLModelClient
import json
import pandas as pd
from datetime import datetime, UTC

In [6]:

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set ploting default
pio.templates.default = "plotly_dark"
px.defaults.height = 600
px.defaults.template = "plotly_dark"
px.defaults.color_continuous_scale = px.colors.sequential.Bluyl_r
px.defaults.color_discrete_sequence = px.colors.sequential.Bluyl_r


plotly_config = {
  "staticPlot": True,
#   "scrollZoom": True,
  "displayModeBar": False,
  "editable": True,
}


# np.random.seed (21)
pd.set_option ("display.max_rows", 20)
pd.set_option ("display.max_columns", 15)
pd.options.plotting.backend = "plotly"

# mute warnings
# warnings.filterwarnings ("ignore")

In [ ]:
database_client = SQLModelClient(database_url="sqlite:////Users/kunmi/Documents/tagging_system/data/database/dev_1.db")

start_date = '2026-01-19'
end_date = '2026-01-27'
with database_client as db:
  res = db.execute(
    f"""select
        a.name,
        [a_snap].*,
        MAX(price) OVER (
          PARTITION BY a_snap.asset_id
          ORDER BY data_date
          ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS recent_high_30d,
        MIN(price) OVER (
          PARTITION BY a_snap.asset_id
          ORDER BY data_date
          ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS recent_low_30d,
        AVG(price) OVER (
          PARTITION BY a_snap.asset_id
          ORDER BY data_date
          ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS ma_30,
        AVG(price) OVER (
          PARTITION BY a_snap.asset_id
          ORDER BY data_date
          ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
        ) AS ma_50,
        t.name as tag_name
      from asset a
      INNER JOIN asset_snapshot as [a_snap]
        on a.id = [a_snap].asset_id
      LEFT JOIN asset_tag as at
        ON a.id = at.asset_id
      LEFT JOIN tag as t
        ON at.tag_id = t.id
    WHERE date(a_snap.data_date) BETWEEN '{start_date}' AND '{end_date}'
      AND a_snap.asset_id IS NOT NULL
  """
  )
  res = res.fetchall()
df = pd.DataFrame(res)
df.columns

Index(['name', 'id', 'asset_id', 'data_date', 'currency', 'local_currency',
       'share', 'price', 'avg_price', 'value', 'cost', 'profit', 'fx_impact',
       'recent_high_30d', 'recent_low_30d', 'ma_30', 'ma_50', 'tag_name'],
      dtype='object')

In [204]:
df.head(3)

,name,id,asset_id,data_date,currency,local_currency,share,...,profit,fx_impact,recent_high_30d,recent_low_30d,ma_30,ma_50,tag_name
0,AMDd_EQ,2302,1,2026-01-19 00:00:00.139439+00:00,EUR,EUR,0.071563,...,3.65,NaN,200.4,200.4,200.4,200.4,Sample Tag_5578
1,AMDd_EQ,2302,1,2026-01-19 00:00:00.139439+00:00,EUR,EUR,0.071563,...,3.65,NaN,200.4,200.4,200.4,200.4,hey
2,AMDd_EQ,2361,1,2026-01-19 00:14:33.953577+00:00,EUR,EUR,0.071563,...,3.65,NaN,200.4,200.4,200.4,200.4,Sample Tag_5578


In [205]:
total_asset = df.loc[:,['name']].value_counts()
total_asset

name       
CBUKd_EQ       1509
AMDd_EQ        1006
EXH1d_EQ       1006
IDVYa_EQ       1006
ISFEl_EQ       1006
IUS3_EQ        1006
ANRJl_EQ        503
CHVd_EQ         503
GOOGL_US_EQ     503
SNII_US_EQ      503
WCBRl_EQ        503
Name: count, dtype: int64

In [237]:
list(df['name'].unique())



['AMDd_EQ',
 'CBUKd_EQ',
 'CHVd_EQ',
 'ISFEl_EQ',
 'SNII_US_EQ',
 'ANRJl_EQ',
 'IUS3_EQ',
 'EXH1d_EQ',
 'IDVYa_EQ',
 'GOOGL_US_EQ',
 'WCBRl_EQ']

In [ ]:

def prep_data(df):
    df = df.copy()
    df['data_datetime'] = pd.to_datetime(df['data_date'])
    df['date'] = df['data_datetime'].dt.date

    import numpy as np
    df['pct_drawdown'] = (df['price'] - df['recent_high_30d']) / df['recent_high_30d']
    df['price_vs_ma_50'] = np.where(
        df['ma_50'] != 0,
        (df['price'] - df['ma_50']) / df['ma_50'],
        None
      )
    df['norm_price_30d'] = (df['recent_high_30d'] - df['recent_low_30d']) / df['ma_30']
    df['volatility_30d'] = (
        df.groupby('asset_id')['price']
          .pct_change()
          .rolling(30)
          .std()
      )
    df['dca_bias'] = (
        -0.5 * df['pct_drawdown']
        -0.4 * df['price_vs_ma_50']
        +0.1 * df['volatility_30d']
    )
    # df = df.drop(['price'], axis=1)
    return df


df = prep_data(df)

In [208]:
# Sort by datetime descending
df_sorted = df.sort_values(['tag_name', 'data_datetime'], ascending=[True, False])

# Keep only the first (latest) per tag_name
latest_per_tag = df_sorted.groupby('tag_name').first().reset_index()

In [209]:
current_profit_per_tag = latest_per_tag.groupby(['name', 'tag_name'])['profit'].sum().reset_index(name='profit')

In [210]:
current_profit_per_tag

,name,tag_name,profit
0,AMDd_EQ,Sample Tag_5578,4.54
1,AMDd_EQ,hey,4.54
2,CBUKd_EQ,Sample Tag_1055,-1.01
3,CBUKd_EQ,Sample Tag_7162,-1.01
4,CBUKd_EQ,akin,-1.01
5,IDVYa_EQ,US,3.31
6,ISFEl_EQ,Core,10.38


In [196]:
fig = px.bar (
  current_profit_per_tag.reset_index (),
  "tag_name",
  "profit",
  # facet_col="Pclass",
  color="profit",
  color_continuous_scale="earth",
#   color_discrete_sequence="Tealgrn_r",
  text="name",
)

fig.update_layout (
  height=500,
)

In [197]:
x

,data_datetime,norm_price_30d,name
450,2026-01-26 11:42:17.498552+00:00,0.000000,ISFEl_EQ
451,2026-01-26 11:42:17.498552+00:00,0.000000,ISFEl_EQ
452,2026-01-26 11:50:58.231400+00:00,0.000316,ISFEl_EQ
453,2026-01-26 11:50:58.231400+00:00,0.000316,ISFEl_EQ
454,2026-01-26 11:57:09.823564+00:00,0.000506,ISFEl_EQ
...,...,...,...
595,2026-01-27 02:21:27.871149+00:00,0.000000,ISFEl_EQ
596,2026-01-27 02:31:29.630884+00:00,0.000000,ISFEl_EQ
597,2026-01-27 02:31:29.630884+00:00,0.000000,ISFEl_EQ
598,2026-01-27 02:41:37.240417+00:00,0.000000,ISFEl_EQ


In [198]:
asset_name = 'ISFEl_EQ'

x = df[df['name'] == asset_name][['data_datetime', 'profit', 'name']].sort_values(by="data_datetime")

fig = px.line (
  x,
  "data_datetime",
  "profit",
#   color_discrete_sequence="Tealgrn_r",
)

fig.update_layout (
  height=500,
)

In [199]:
dfasset_name = 'ISFEl_EQ'

x = df[df['name'] == asset_name][['data_datetime', 'pct_drawdown', 'name']].sort_values(by="data_datetime")

fig = px.line (
  x,
  "data_datetime",
  "pct_drawdown",
#   color_discrete_sequence="Tealgrn_r",
)

fig.update_layout (
  height=500,
)

In [212]:
asset_name = 'AMDd_EQ'

x = df[df['name'] == asset_name][['data_datetime', 'ma_30', 'ma_50', 'name']].sort_values(by="data_datetime")

fig = px.line (
  x,
  "data_datetime",
  ["ma_30", "ma_50"],
#   color_discrete_sequence="Tealgrn_r",
)

fig.update_layout (
  height=500,
)

In [215]:
current_profit_per_tag

,name,tag_name,profit
0,AMDd_EQ,Sample Tag_5578,4.54
1,AMDd_EQ,hey,4.54
2,CBUKd_EQ,Sample Tag_1055,-1.01
3,CBUKd_EQ,Sample Tag_7162,-1.01
4,CBUKd_EQ,akin,-1.01
5,IDVYa_EQ,US,3.31
6,ISFEl_EQ,Core,10.38


In [219]:
current_profit_per_tag_2 = latest_per_tag.groupby(['tag_name', 'name'])['profit'].sum().reset_index(name='profit')
current_profit_per_tag_2

,tag_name,name,profit
0,Core,ISFEl_EQ,10.38
1,Sample Tag_1055,CBUKd_EQ,-1.01
2,Sample Tag_5578,AMDd_EQ,4.54
3,Sample Tag_7162,CBUKd_EQ,-1.01
4,US,IDVYa_EQ,3.31
5,akin,CBUKd_EQ,-1.01
6,hey,AMDd_EQ,4.54


In [228]:
tag_name = 'hey'

subset = current_profit_per_tag[current_profit_per_tag['tag_name'] == tag_name]

fig = px.pie (
  values=subset.profit,
  labels=subset.name,
  hover_name=subset.name,
  # hover_data=subset.profit

)

# fig.show(config=plotly_config)

fig.show()